# 10 Case Study: An HR-Facing Demo

## 用一个业务化任务证明这套系统不是概念拼贴

到这一章，前面的理论结构已经基本齐了：模型为什么能表现出任务性，system prompt 和 roles 如何构成控制面，tool calling 如何把自然语言决策压缩成结构化动作，Agent 和 MCP 分别负责什么，本地模型和本地能力层如何接起来，Runtime 又如何把这一切组织成闭环。问题来了：这些结构如果不被放进一个完整任务里，读者最终看到的仍然只是概念分层，而不是系统成立。

所以这一章不再继续解释机制，而是直接做一件对 HR 也能读懂、但又不会显得幼稚的事情：让系统围绕一个招聘相关的实际任务跑一遍完整闭环。

选择招聘场景有两个好处。第一，它天然适合这个项目的展示对象，因为 HR、技术经理、面试官都能迅速理解问题价值。第二，它能同时考验这套系统最关键的几种能力：读取文档、抽取结构化要求、比对候选人信息、输出可执行判断，而不是仅仅生成一段漂亮但空泛的总结。

这意味着，本章真正要证明的不是“模型能分析简历”，而是：**前面搭起来的 Agent Runtime 与 MCP 能力层，是否真的能在一个有业务味的任务里协同工作。**

## 先给结论

如果把这一章的目标压缩成一句话，可以这样说：

> 一个像样的案例演示，不应该只是模型给出一段看起来专业的结论，而应该让读者清楚看到：系统为什么读了哪些材料、做了哪些动作、基于哪些证据形成了最后判断。

这句话决定了整章的展示方式：

- 重点不是最终答案有多像人写的
- 重点是中间过程是否体现出 Runtime 的组织能力
- 重点是能力层是否真的被消费，而不是摆设
- 重点是系统输出是否带有可追溯证据，而不是纯粹“模型觉得”

换句话说，这一章的 demo 必须像一个系统在工作，而不是像一个模型在表演。

## 1. 为什么选“JD 与候选人能力映射”作为案例

案例选择对整个项目的观感影响很大。选得太技术化，HR 看不出价值；选得太浅，又会显得像课堂练习。`JD 与候选人能力映射` 正好落在一个比较平衡的位置上。

它至少具备几个优点：

- 业务目标清晰：判断岗位需求与候选人经历是否匹配
- 对外部材料依赖强：不能只靠模型臆测，必须读取 JD 和候选人资料
- 结构化输出天然成立：能力维度、证据、风险点、追问建议都可以组织成结果
- 对招聘场景足够贴近：HR 和技术面试官都能理解这个任务的现实价值

更重要的是，这个任务天然能把前面几章的几个关键能力都调动起来：resource 读取、tool 调用、状态推进、证据回填和最终综合判断。也正因为如此，它适合作为这套系统的第一支完整案例。

## 2. 这个案例真正要解决的问题是什么

如果把任务说得太宽，比如“帮我分析这个候选人合不合适”，系统很容易退回泛泛而谈的职业建议。更好的做法是把任务定义得更像一个可执行流程：

- 读取岗位 JD
- 读取候选人背景材料
- 抽取岗位的核心能力维度
- 将候选人经历映射到这些维度上
- 输出结构化结论，包括证据、缺口和面试追问点

这个定义非常关键。因为它把一个模糊的判断题，改造成了一个有输入、有步骤、有输出标准的任务结构。这样，Runtime 才能真正发挥作用，而不是只是把一句模糊问题丢给模型。

## 3. 为什么这个任务不能直接让模型“看着办”

如果只是做一个聊天 demo，完全可以直接把 JD 和候选人材料粘进同一个 prompt，让模型输出一段匹配分析。这样甚至常常也能得到看起来挺像样的结果。

但这种做法的问题在于，它几乎无法证明系统结构：

- 无法证明模型是否真的区分了岗位要求和候选人证据
- 无法证明能力抽取是不是靠显式动作完成的
- 无法证明结果里的结论是否可追溯到具体材料
- 无法证明 Runtime 是否真的在维护任务逻辑

换句话说，直接塞进大 prompt 得到一个漂亮结果，只能证明模型能生成一段分析文字，不能证明你搭了一个有结构的 Agent 系统。对这套项目来说，这显然不够。

## 4. 这个案例的理想执行路径应该是什么样

从 Runtime 视角看，一个像样的执行路径大概应该长这样：

1. Runtime 接收任务目标：生成岗位与候选人的匹配分析
2. Runtime 暴露当前可用 resources、tools 和 prompts
3. 模型先选择读取 JD resource
4. 模型再选择读取候选人资料 resource
5. Runtime 触发要求抽取类 tool，提炼岗位能力维度
6. Runtime 再触发匹配评分类 tool，形成结构化中间结果
7. 模型基于这些证据输出最终综合分析

这条路径的意义不在于“步骤多”，而在于它能明确展示系统每一步都在干什么。特别是对 HR 或非技术读者来说，他们未必关心协议细节，但会很在意系统到底是不是基于真实材料得出的判断。

把上面那条理想路径再压得更具体一点，会更像一个真的运行轨迹：第一步不是分析，而是先把 JD 读进来，因为系统必须先知道岗位到底要什么；第二步才是读取候选人资料，因为没有证据对象，后面的匹配就只能空转；第三步调用 `extract_key_requirements`，把 JD 里的自然语言要求压成几个明确维度，比如 LLM 系统理解、tool use、agent runtime 设计；第四步调用 `score_candidate_fit`，不是为了得到一个神秘总分，而是为了把“候选人的哪段经历对应哪项要求”显式对齐出来；最后模型才有资格去生成总结，因为这时它面对的已经不是两段原始材料，而是一组被整理过的判断依据。

这一段之所以值得单独写出来，是因为它让读者看到：这个案例不是“模型读两段材料后写了篇分析”，而是先读材料，再做抽取，再做映射，最后才综合表达。换句话说，最后那段输出不是起点，而是前面几步动作的结果。

## 5. Resource 在这个案例里承担什么职责

这个案例最能体现 MCP 价值的一点，就是它不是一上来就调用工具，而是先读取材料。因为招聘判断不是一个计算题，而是一个证据组织题。

这里的 resources 至少可以包括：

- 岗位 JD 文本
- 候选人经历摘要
- 可选：团队对岗位的补充说明

这些 resources 的价值不是“提供背景资料”这么简单，而是它们决定了后面的所有判断有没有根。模型如果不先读这些资源，后面的能力抽取和匹配分析就会不可避免地带有大量凭空补全。

因此，从这个案例看，resource 层不是 MCP 中一个看起来优雅但可有可无的对象，它是整套判断是否可信的第一道基础。

## 6. Tool 在这个案例里承担什么职责

工具层在这个案例中的作用，不是为了显得系统更高级，而是为了把某些关键动作从“模型直觉判断”变成“显式步骤”。

例如：

- `extract_key_requirements` 负责把 JD 中的要求显式提取出来
- `score_candidate_fit` 负责把候选人经历与这些要求建立结构化映射
- 可选的 `summarize_section` 负责压缩过长材料，减少后续上下文负担

这些工具的价值在于，它们把系统过程拆得更透明。即使最终判断仍然需要模型综合生成，读者也能看清哪些部分是显式结构化处理过的，哪些部分是最后的综合表达。

## 7. Prompt Template 在这个案例里为什么也有必要

如果没有任务模板，这个案例很容易被模型处理成泛泛的“简历点评”。有了合适的 prompt 模板，Runtime 才能把模型导入一个更严格的分析框架。

例如，`map_candidate_to_jd` 这样的 prompt 可以预先规定：

- 输出必须按能力维度组织
- 每条判断都应尽量给出来自材料的证据依据
- 缺口和风险要单独列出
- 如果材料不足，应该承认空白而不是补想象

这样做的效果，是把模型从“自由发表职业意见”的轨道，推回到“基于材料生成结构化评估”的轨道。对这个案例而言，这一步非常关键。

## 8. 最终输出不该只是结论，而应该像一个可消费结果

如果这个案例最后只输出一句“这个候选人挺合适”，那即使过程再复杂，展示价值也会掉得很快。更好的结果应该是能被招聘流程继续消费的结构化交付物。

例如，最终输出可以明确分成：

- 核心匹配结论
- 关键能力维度及证据
- 不足或风险点
- 建议在面试中进一步验证的问题

这种结果比一段漂亮长文更有说服力，因为它显得像招聘流程中的工作产物，而不是模型的文风展示。对 HR 来说，这种“结果可继续使用”的感觉会比模型多会说话更重要。

如果把“最终交付长什么样”说得更实一点，它至少应该能被拆成四块。第一块是总体判断，但这个判断不能只是 `Strong / Moderate / Weak` 这种标签本身，而应该顺手解释一句为什么是这个等级。第二块是能力维度，每个维度下面都应该有证据和缺口，例如“候选人确实做过 MCP 相关项目”属于证据，“只提到了调用工具但没有体现状态治理”属于缺口。第三块是风险项，这里不是重复说缺点，而是把那些目前材料里无法确认、但岗位又非常在意的地方单独拎出来。第四块是面试追问点，它要像真的能拿去面试里问的问题，比如“你在多步任务里怎么处理工具调用失败后的状态回滚”，而不是泛泛的“再深入聊聊工程经验”。

这一点很关键，因为它决定了这个案例最后产出的到底是一篇 prose，还是一份工作结果。前者只能证明模型会写，后者才说明系统真的把材料、动作和结论组织成了一个可继续被招聘流程消费的产物。

## 9. 为什么这个案例对 HR 友好，但又不会显得太浅

一个很难拿捏的点是：既然项目给 HR 看，那就不能太晦涩；但如果为了“友好”把案例做成简历打分小游戏，又会立刻失去技术质感。

这个案例之所以比较合适，是因为它同时满足两点：

- 从业务语言看，它非常直观，HR 一眼就知道在做什么
- 从系统结构看，它又足够复杂，能真实调动 Resource、Tool、Prompt、Runtime 和模型多轮推理

也就是说，案例的友好度来自业务语境，而不是来自技术降级。真正的复杂度并没有被拿掉，只是被放进了一个更容易理解的任务里。

## 10. 这个案例真正证明了什么

如果这个案例跑得通，它真正证明的不是“你能做招聘分析”。更重要的是它在结构上证明了几件事：

- 你搭的 MCP 能力层不是摆设，resource、tool、prompt 都被真正消费了
- 你定义的 Agent Runtime 不是概念，它能把任务组织成多步闭环
- 你用本地模型做的不是聊天，而是受约束的任务推进
- 你输出的不是漂亮 prose，而是可继续被流程使用的结构化结果

从展示角度看，这些比单纯展示一段高质量文案要有分量得多。因为它们对应的是系统判断力，而不是文风模仿能力。

## 11. 这个案例也有边界，边界说明系统成熟度

一个好的案例说明，不应该只展示流程顺滑的一面，也应该承认它的边界在哪里。比如：

- 如果 JD 写得太模糊，能力抽取结果本身就会不稳定
- 如果候选人资料过度营销化，证据映射会被噪音污染
- 如果模型在长上下文下丢失某些约束，输出结构可能会漂移

把这些边界说明清楚，不会削弱 demo，反而会提升可信度。因为读者能看出这里不是在假装系统无所不能，而是在承认系统依赖怎样的输入质量和运行条件。

## 12. 为什么案例之后必须接失败模式与评估

到这一章结束，系统已经在一个完整任务里闭环了。但如果项目在这里停下，仍然会留下一个隐患：读者看到的是一条理想路径，却还不知道这条路径在失败时会怎样、质量该如何评估、哪些风险需要 guardrail。

所以，下一章最自然的动作不是换一个案例，而是回头做系统审视：

- 这条闭环最容易在哪些环节出错
- 出错后 Runtime 应该如何反应
- 如何知道这个系统是真的在用能力层，而不是只是在写一篇更长的回答

也就是说，案例证明系统能工作，评估章节证明系统不是只在理想状态下工作。

## 13. 本章结论

这一章最值得保留的判断有这些：

- 好的案例不是让模型说得漂亮，而是让系统过程变得可见、可追溯、可解释。
- `JD 与候选人能力映射` 之所以合适，是因为它同时具备业务可理解性和系统结构复杂度。
- 这个案例里 resource 负责证据基础，tool 负责显式动作，prompt 负责任务入口，Runtime 负责把它们串成闭环。
- 最终输出应该像可被招聘流程消费的结果，而不是一段漂亮 prose。
- 真正有说服力的 demo，不是只展示成功答案，而是为下一步的失败模式与评估留出空间。

下一章会专门处理这个空间：系统可能如何失败，如何加 guardrail，以及怎样评估它到底有没有真正完成一个 Agent 系统该完成的事情。